In [ ]:
import numpy as np
from astropy.io import fits
from astropy.table import Table
from datetime import datetime
import copy
import collections
from astropy.cosmology import Planck13  # Planck (2014) Paper XVI, Table 5

import gc
gc.enable()

import sys
sys.path.append('..') 

# from utils import Vcom, get_comoving_distance_h, c, z_min
from utils import get_comoving_distance_h, c, z_min, get_comoving_distance, cosmo
# from data_stat_func import wprp, dPdN
from data_stat_func import weighted_wprp
from data_stat_func import wprp

from halotools.mock_observables import apply_zspace_distortion
from halotools.mock_observables import return_xyz_formatted_array
import halotools.mock_observables as htmo

import Corrfunc
from Corrfunc.mocks.DDrppi_mocks import DDrppi_mocks
from Corrfunc.utils import convert_rp_pi_counts_to_wp
import matplotlib.pyplot as plt
import copy

# from Ng8_func import N_g_8, N_g_8_theory, select_bottom20_in_random, select_top20_in_random
from Ng8_func import N_g_8_cross, N_g_8_theory_cross, select_bottom20_in_random, select_top20_in_random

In [ ]:
z_max_arr  = np.array((       47650, 31900, 19250, 12500))/ c
M_thr_arr  = np.array((-22.0, -21.0, -20.0, -19.0, -18.0))
C_thr_arr  = np.array([[0.0,  0.82,  0.94,  0.97,   1.2,],
                       [0.0,  0.72,  0.90,  0.94,   1.2,],
                       [0.0,  0.55,  0.85,  0.92,   1.2,],
                       [0.0,  0.50,  0.73,  0.87,   1.2,],])

M_thr_list  = [ 'Mr_22_21', 'Mr_21_20', 'Mr_20_19', 'Mr_19_18', ]
C_thr_list  = np.array([['0_p82'  , 'p82_p94', 'p94_p97', 'p97_1p2',],
                        ['0_p72'  , 'p72_p90', 'p90_p94', 'p94_1p2',],
                        ['0_p55'  , 'p55_p85', 'p85_p92', 'p92_1p2',],
                        ['0_p50'  , 'p50_p73', 'p73_p87', 'p87_1p2',],])

Mr_thr_text = [r'$M_{\rm r}: [-22,-21]$',r'$M_{\rm r}: [-21,-20]$',r'$M_{\rm r}: [-20,-19]$',r'$M_{\rm r}: [-19,-18]$',]
Co_thr_text = np.array([[r'g-r: [0,0.82]', r'g-r: [0.82,0.94]', r'g-r: [0.94,0.97]', r'g-r: [0.97,1.2]',],
                        [r'g-r: [0,0.72]', r'g-r: [0.72,0.90]', r'g-r: [0.90,0.94]', r'g-r: [0.94,1.2]',],
                        [r'g-r: [0,0.55]', r'g-r: [0.55,0.85]', r'g-r: [0.85,0.92]', r'g-r: [0.92,1.2]',],
                        [r'g-r: [0,0.50]', r'g-r: [0.50,0.73]', r'g-r: [0.73,0.87]', r'g-r: [0.87,1.2]',],])

In [ ]:
# Boundary limited
def Vcom(zmax, angarea = 6369):
    return (4 * np.pi / 3 * (get_comoving_distance(zmax) ** 3 - get_comoving_distance(0.02) ** 3)
            * cosmo.h ** 3 * angarea / 41253)

In [ ]:
### Comoving pi_max and wp bins in Mpc/h
# rp_bins = np.logspace(-0.8, 1.6, 13)
# rp_bins_x = (rp_bins[1:] + rp_bins[:-1]) / 2
Mr_bins = np.linspace(-22, -18, 21)
Mr_bins_x = (Mr_bins[:-1] + Mr_bins[1:])/2
# Mr_bins_x

from matplotlib import cm

colors4 = [cm.YlGnBu(55)  , cm.YlGnBu(80)  , cm.YlGnBu(120) , cm.YlGnBu(150) , cm.YlGnBu(180)]

In [ ]:
Mr_bins_x

In [ ]:
data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1

plt.figure(figsize=(15, 3))
plt.scatter(data['ra'], data['dec'], s = 0.1)

In [ ]:
from scipy.optimize import fsolve

def f1(x):
    """Western limit: δ < -2.555556(α - 131°)"""
    return -2.555556 * (x - 131)

def f2(x):
    """Northern limit: δ < arcsin[...] - 更简洁的版本"""
    angle_rad = np.deg2rad(x - 95.9)
    
    numerator = 0.93232 * np.sin(angle_rad)
    denominator = np.sqrt(1 - (0.93232 * np.cos(angle_rad))**2)
    
    # 使用np.where处理分母为零的情况
    argument = np.where(denominator != 0, numerator / denominator, 0)
    argument = np.clip(argument, -1, 1)
    
    return np.rad2deg(np.arcsin(argument))

def f3(x):
    """Eastern limit: δ < 1.70909(α - 235°)"""
    return 1.70909 * (x - 235)

def equation1(x):
    return f1(x) - f2(x)

def equation2(x):
    return f3(x) - f2(x)
# 求解交点
# 提供多个初始猜测值以找到所有交点
initial_guesses1 = [115]
intersections1 = []
initial_guesses2 = [255]
intersections2 = []
for guess in initial_guesses1:
    root = fsolve(equation1, guess)
    # 去除重复的解（考虑浮点精度）
    if not any(np.isclose(root, x, atol=1e-5) for x in intersections1):
        intersections1.append(root[0])
for guess in initial_guesses2:
    root = fsolve(equation2, guess)
    # 去除重复的解（考虑浮点精度）
    if not any(np.isclose(root, x, atol=1e-5) for x in intersections2):
        intersections2.append(root[0])

print("交点 x 坐标:", intersections1)
print("交点坐标:")
for x in intersections1:
    y = f1(x)  # 或者 f2(x)，在交点处相等
    print(f"({x:.4f}, {y:.4f})")

print("交点 x 坐标:", intersections2)
print("交点坐标:")
for x in intersections2:
    y = f3(x)  # 或者 f2(x)，在交点处相等
    print(f"({x:.4f}, {y:.4f})")

In [ ]:
def boundary_limit(stat_ra, stat_dec, ):
    # section west
    ra_mask  = (115.1858 < stat_ra) & (stat_ra < 131)
    dec_mask = (f1(stat_ra) < stat_dec) & (stat_dec < f2(stat_ra))
    western_mask = ra_mask & dec_mask
    # section middle
    ra_mask  = (131 < stat_ra) & (stat_ra < 235)
    dec_mask = (0 < stat_dec) & (stat_dec < f2(stat_ra))
    middle_mask = ra_mask & dec_mask
    # section east
    ra_mask  = (235 < stat_ra) & (stat_ra < 257.7142)
    dec_mask = (f3(stat_ra) < stat_dec) & (stat_dec < f2(stat_ra))
    eastern_mask = ra_mask & dec_mask
    # total mask
    boundary_mask = western_mask | middle_mask | eastern_mask

    return boundary_mask

In [ ]:
plt.figure(figsize=(12, 6))

plt.scatter(intersections1, [f1(x) for x in intersections1], 
           color='red', s=50, zorder=5)#, label='intersections1'
plt.scatter(intersections2, [f3(x) for x in intersections2], 
           color='red', s=50, zorder=5)#, label='intersections2'
# 绘制函数进行比较
alpha_values = np.linspace(100, 300, 200)


# 绘制三个边界函数
plt.plot(alpha_values, f1(alpha_values), 'r-', label='Western: δ < -2.555556(α-131)', linewidth=2)
plt.plot(alpha_values, f2(alpha_values), 'b-', label='Northern: δ < arcsin[...]', linewidth=2)
plt.plot(alpha_values, f3(alpha_values), 'g-', label='Eastern: δ < 1.70909(α-235)', linewidth=2)

# 添加水平线 δ = 0 (southern limit)
plt.axhline(y=0, color='purple', linestyle='-', label='Southern: δ > 0°')

### SDSS plot 
data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1

boundary_mask = boundary_limit(data['ra'], data['dec'])

plt.scatter(data['ra'][boundary_mask], data['dec'][boundary_mask], s = 0.1)

# 设置图表属性
plt.xlabel(r'$\alpha (°)$')
plt.ylabel(r'$\delta (°)$')
plt.title('Boundary Limits Comparison')
plt.legend(fontsize = 14)
plt.grid(True, alpha=0.3)
plt.xlim(80, 300)
plt.ylim(-20, 90)

plt.show()

### $N_g^8$ percentile $w_p(r_p)$

In [ ]:
z_Mr = np.array([[-22, 73500],[-21.5, 59600], [-21, 47650], [-20.5, 39700], [-20, 31900], [-19.5, 25450], [-19, 19250], [-18.5, 15750],[-18,12500]])


In [ ]:
import numpy as np
from astropy.cosmology import Planck18, FlatLambdaCDM, z_at_value
from astropy import units as u

def redshift_from_magnitude(M_abs, m_app=17.6, cosmology=Planck13):
    """
    通过视星等和绝对星等计算红移。

    参数:
    m_app (float): 视星等
    M_abs (float): 绝对星等
    cosmology: 宇宙学模型，默认为Planck18

    返回:
    float: 红移 z
    """
    # 1. 计算距离模数
    distance_modulus = m_app - M_abs
    
    # 2. 计算光度距离 (单位：秒差距)
    d_L_pc = 10 ** ((distance_modulus + 5) / 5)
    
    # 3. 转换为Astropy单位（兆秒差距更常用）
    # d_L = d_L_pc * u.pc
    # 或者更常用 Mpc
    d_L_mpc = (d_L_pc / 1e6) * u.Mpc / cosmology.h
    
    # 4. 使用宇宙学模型求解红移z
    #    z_at_value 函数数值求解：cosmology.luminosity_distance(z) = d_L
    try:
        z = z_at_value(cosmology.luminosity_distance, d_L_mpc)
        return z
    except Exception as e:
        print(f"求解失败: {e}")
        return None

# 示例1：近距离天体（z ≈ 0）
plt.figure(figsize=(6, 6),dpi=100)

M1 = np.linspace(-18, -22, 21)
M1_x = (M1[1:] + M1[:-1])/2

z1 = np.zeros(len(M1_x))
for i in range(len(M1_x)):
    z1[i] = redshift_from_magnitude(M1_x[i])

z_thr = np.zeros(len(M1))
for i in range(len(M1)):
    z_thr[i] = redshift_from_magnitude(M1[i])

plt.plot(z1, M1_x, color='green')
plt.plot(z_Mr[:,1]/300000, z_Mr[:,0], color='gray')

plt.plot([0.02, 0.02], [-18, -22], lw = 2, c = 'black')
plt.plot([0.02, redshift_from_magnitude(-19)], [-18, -18], lw = 2, c = 'black')
plt.plot([0.02, redshift_from_magnitude(-19)], [-22, -22], lw = 2, c = 'black')
plt.plot([redshift_from_magnitude(-19), redshift_from_magnitude(-19)], [-18, -22], lw = 2, c = 'black', label = 'Ng8')

plt.plot([0.02, 0.02], [-19, -22], lw = 2, ls = '--', c = 'red')
plt.plot([0.02, redshift_from_magnitude(-19)], [-19, -19], lw = 2, ls = '--', c = 'red')
plt.plot([0.02, redshift_from_magnitude(-19)], [-22, -22], lw = 2, ls = '--', c = 'red')
plt.plot([redshift_from_magnitude(-19), redshift_from_magnitude(-19)], [-19, -22], lw = 2, ls = '--', c = 'red', label = 'DDP')

plt.plot([0.02, z_thr[0]], [M1[0], M1[0]], lw = 2, c = 'royalblue', label = 'env LF volume')

for i in range(5):
    plt.plot([0.02, z_thr[i]], [M1[i], M1[i]], lw = 2, c = 'royalblue')
    plt.plot([z_thr[i], z_thr[i]], [M1[i], M1[i+1]], lw = 2, c = 'royalblue')

plt.legend()

plt.xlabel('redshift')
plt.ylabel('Mr')
plt.xlim(0.01,0.25)
plt.gca().invert_yaxis()

In [ ]:
M1 = np.linspace(-18, -22, 21)
M1

In [ ]:
def env_LF(gal_Mr, gal_gmr, gal_z, Mr_brighter, Mr_fainter):
    red_galaxies_mask  = gal_gmr > (0.21 - 0.03*gal_Mr)
    blue_galaxies_mask = gal_gmr < (0.21 - 0.03*gal_Mr)
    Mr_bin_width       = Mr_fainter - Mr_brighter

    Mr_mask = (Mr_brighter < gal_Mr) & (gal_Mr < Mr_fainter)
    z_max   = redshift_from_magnitude(Mr_fainter) # z_max depends on the the fainter Mr
    z_mask  = (z_min < gal_z) & (gal_z < z_max)
    Phi_LF_red  = np.count_nonzero(red_galaxies_mask  & Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width
    Phi_LF_blue = np.count_nonzero(blue_galaxies_mask & Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width

    return Phi_LF_red, Phi_LF_blue

In [ ]:
def env_LF_DDP(gal_Mr, gal_gmr, gal_z, Mr_brighter, Mr_fainter, DDP_Mr):
    red_galaxies_mask  = gal_gmr > (0.21 - 0.03*gal_Mr)
    blue_galaxies_mask = gal_gmr < (0.21 - 0.03*gal_Mr)
    Mr_bin_width       = Mr_fainter - Mr_brighter

    Mr_mask = (Mr_brighter < gal_Mr) & (gal_Mr < Mr_fainter)
    # z_max   = redshift_from_magnitude(Mr_fainter) # z_max depends on the the fainter Mr
    z_max   = redshift_from_magnitude(DDP_Mr) # z_max depends on the the fainter Mr
    z_mask  = (z_min < gal_z) & (gal_z < z_max)
    Phi_LF_red  = np.count_nonzero(red_galaxies_mask  & Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width
    Phi_LF_blue = np.count_nonzero(blue_galaxies_mask & Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width

    return Phi_LF_red, Phi_LF_blue

In [ ]:
def env_LF_theory(gal_Mr, gal_gmr, Mr_bins):
    red_galaxies_mask  = gal_gmr > (0.21 - 0.03*gal_Mr)
    blue_galaxies_mask = gal_gmr < (0.21 - 0.03*gal_Mr)
    Mr_bin_width       = Mr_bins[1] - Mr_bins[0]

    Phi_LF_red  = np.zeros(len(Mr_bins)-1)
    Phi_LF_blue = np.zeros(len(Mr_bins)-1)

    for i in range(len(Mr_bins)-1):
        Mr_mask = (Mr_bins[i] < gal_Mr) & (gal_Mr < Mr_bins[i+1])
        Phi_LF_red[i]  = np.count_nonzero(red_galaxies_mask  & Mr_mask) / 1000**3 / Mr_bin_width
        Phi_LF_blue[i] = np.count_nonzero(blue_galaxies_mask & Mr_mask) / 1000**3 / Mr_bin_width

    return Phi_LF_red, Phi_LF_blue

In [ ]:
# def Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, gal_fac, Mr_bins):
def Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, Mr_bins, DDP_Mr = -19):

    env_LF_bottom20_red  = np.zeros(len(Mr_bins)-1)
    env_LF_bottom20_blue = np.zeros(len(Mr_bins)-1)
    env_LF_top20_red     = np.zeros(len(Mr_bins)-1)
    env_LF_top20_blue    = np.zeros(len(Mr_bins)-1)

    ### Ng8 ############################################################################################################
    # thr_mask    = gal_Mr < Mr_bins[i+1] # fainter end
    # # thr_mask    = (gal_Mr < Mr_bins[i+1]) & (gal_fac >= 0.9) # fainter end, f_ac completenesss (r = 8 Mpc/h) >= 0.9
    # z_max       = redshift_from_magnitude(Mr_bins[i+1]) # z_max depends on the the fainter Mr
    # z_mask      = (gal_z > z_min) & (gal_z < z_max)
    # # pos_mask    = (100 < gal_pos[:, 0]) & (gal_pos[:, 0] < 300)
    # mask        = thr_mask & z_mask #& pos_mask

    mask        = (-22 < gal_Mr) & (gal_Mr < -18) & (0.02 < gal_z) & (gal_z < redshift_from_magnitude(DDP_Mr))

    gal_pos_thr = gal_pos[mask]
    gal_Mr_thr  = gal_Mr[mask]
    gal_gmr_thr = gal_gmr[mask]
    gal_z_thr   = gal_z[mask]
    # gal_fac_thr = gal_fac[mask]

    gal_pos_DDP = gal_pos[(-22 < gal_Mr) & (gal_Mr < DDP_Mr) & (0.02 < gal_z) & (gal_z < redshift_from_magnitude(DDP_Mr))]

    Ng8_thr_DDP = N_g_8_cross(gal_pos_thr, gal_pos_DDP)
    # Ng8_thr    = N_g_8( D1_ra_thr, D1_dec_thr, D1_z_thr ) / gal_fac_thr

    ######################################################################################################################
    # cross correlation with data and data(Ng8 bottom 20%)
    D2_bot_Mr  = select_bottom20_in_random(gal_Mr_thr , Ng8_thr_DDP)
    D2_bot_gmr = select_bottom20_in_random(gal_gmr_thr, Ng8_thr_DDP)
    D2_bot_z   = select_bottom20_in_random(gal_z_thr  , Ng8_thr_DDP)

    # cross correlation with data and data(Ng8 top 20%)
    D2_top_Mr  = select_top20_in_random(gal_Mr_thr , Ng8_thr_DDP)
    D2_top_gmr = select_top20_in_random(gal_gmr_thr, Ng8_thr_DDP)
    D2_top_z   = select_top20_in_random(gal_z_thr  , Ng8_thr_DDP)
    ######################################################################################################################
    for i in range(len(Mr_bins)-1):
        if i < np.where(Mr_bins == DDP_Mr)[0][0] + 1: # Mr [-22, -19]
            env_LF_bottom20_red[i], env_LF_bottom20_blue[i] = env_LF_DDP(D2_bot_Mr, D2_bot_gmr, D2_bot_z, Mr_bins[i], Mr_bins[i+1], DDP_Mr)
            env_LF_top20_red[i], env_LF_top20_blue[i] = env_LF_DDP(D2_top_Mr, D2_top_gmr, D2_top_z, Mr_bins[i], Mr_bins[i+1], DDP_Mr)

        else: # Mr [-19, -18]
            env_LF_bottom20_red[i], env_LF_bottom20_blue[i] = env_LF(D2_bot_Mr, D2_bot_gmr, D2_bot_z, Mr_bins[i], Mr_bins[i+1])
            env_LF_top20_red[i], env_LF_top20_blue[i] = env_LF(D2_top_Mr, D2_top_gmr, D2_top_z, Mr_bins[i], Mr_bins[i+1])
    ######################################################################################################################
    return env_LF_bottom20_red, env_LF_bottom20_blue, env_LF_top20_red, env_LF_top20_blue

In [ ]:
def Ng8_percentile_env_LF_theory(gal_pos, gal_Mr, gal_gmr, Mr_bins, DDP_Mr = -19):
    # mask        = gal_Mr == gal_Mr
    mask        = (-22 < gal_Mr) & (gal_Mr < -18)
    gal_pos_thr = gal_pos[mask]
    gal_pos_Mr  = gal_Mr[mask]
    gal_pos_gmr = gal_gmr[mask]

    gal_pos_DDP = gal_pos[(-22 < gal_Mr) & (gal_Mr < DDP_Mr)]

    # Ng8_thr = N_g_8_theory( D1_x, D1_y, D1_z )
    Ng8_thr_DDP = N_g_8_theory_cross(gal_pos_thr, gal_pos_DDP)
    ######################################################################################################################
    # cross correlation with data and data(Ng8 bottom 20%)
    D2_Mr  = select_bottom20_in_random(gal_pos_Mr , Ng8_thr_DDP)
    D2_gmr = select_bottom20_in_random(gal_pos_gmr, Ng8_thr_DDP)

    env_LF_bottom20_red, env_LF_bottom20_blue = env_LF_theory(D2_Mr, D2_gmr, Mr_bins)
    ######################################################################################################################
    # cross correlation with data and data(Ng8 top 20%)
    D2_Mr  = select_top20_in_random(gal_pos_Mr , Ng8_thr_DDP)
    D2_gmr = select_top20_in_random(gal_pos_gmr, Ng8_thr_DDP)

    env_LF_top20_red, env_LF_top20_blue = env_LF_theory(D2_Mr, D2_gmr, Mr_bins)
    ######################################################################################################################
    return env_LF_bottom20_red, env_LF_bottom20_blue, env_LF_top20_red, env_LF_top20_blue

#### Evironment dependent LF, Mr_C_bins

In [ ]:
### CCMD 32 Quads
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

for var in vars:
    for quad in quads:
        data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data
        boundary_mask = boundary_limit(data['ra'], data['dec'])
        data = data[boundary_mask]

        gal_pos = np.array([np.array(data['ra']),
                            np.array(data['dec']),    
                            np.array(data['z'])]).T
        gal_Mr  = np.array(data['Mr'])
        gal_gmr = np.array(data['gmr'])
        gal_z   = np.array(data['z'])
        # gal_fac = np.array(data['completeness'])

        env_LF_bottom20_red, env_LF_bottom20_blue, \
        env_LF_top20_red   , env_LF_top20_blue     = Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, Mr_bins)
        # env_LF_top20_red   , env_LF_top20_blue     = Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, gal_fac, Mr_bins)

        np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_bottom20_red_env_LF.npy', env_LF_bottom20_red)
        np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_bottom20_blue_env_LF.npy', env_LF_bottom20_blue)
        np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_top20_red_env_LF.npy', env_LF_top20_red)
        np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_top20_blue_env_LF.npy', env_LF_top20_blue)

In [ ]:
# data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1
# boundary_mask = boundary_limit(data['ra'], data['dec'])
# data = data[boundary_mask]
data = fits.open('../SDSS_bright1_boundary_limit_processed_catalog.fits')[1].data # SDSS bright1

gal_pos = np.array([np.array(data['ra']),
                    np.array(data['dec']),    
                    np.array(data['z'])]).T
gal_Mr  = np.array(data['Mr'])
gal_gmr = np.array(data['gmr'])
gal_z   = np.array(data['z'])
# gal_fac = np.array(data['completeness'])

env_LF_bottom20_red, env_LF_bottom20_blue, \
env_LF_top20_red   , env_LF_top20_blue     = Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, Mr_bins)
# env_LF_top20_red   , env_LF_top20_blue     = Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, gal_fac, Mr_bins)

np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/SDSS_bright1_Mr_C_Ng8_bottom20_red_env_LF.npy', env_LF_bottom20_red)
np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/SDSS_bright1_Mr_C_Ng8_bottom20_blue_env_LF.npy', env_LF_bottom20_blue)
np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/SDSS_bright1_Mr_C_Ng8_top20_red_env_LF.npy', env_LF_top20_red)
np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/SDSS_bright1_Mr_C_Ng8_top20_blue_env_LF.npy', env_LF_top20_blue)

In [ ]:
# output_dict    = collections.defaultdict(list)
# output_dict_jk = collections.defaultdict(list)

# data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1
# # data = fits.open('./CCMD_SDSS_Ng8_fits/SDSS_bright1_processed_catalog_Ng8_completeness.fits')[1].data # SDSS bright1

# gal_pos = np.array([np.array(data['ra']),
#                     np.array(data['dec']),    
#                     np.array(data['z'])]).T
# gal_Mr  = np.array(data['Mr'])
# gal_gmr = np.array(data['gmr'])
# gal_z   = np.array(data['z'])
# # gal_fac = np.array(data['completeness'])

# n_jk = 100
# stat_jk_top_red = np.zeros([n_jk, 20])
# stat_jk_top_blue = np.zeros([n_jk, 20])
# stat_jk_bottom_red = np.zeros([n_jk, 20])
# stat_jk_bottom_blue = np.zeros([n_jk, 20])
            
# for k in range(n_jk):
#     jk_mask = data['jk_label'] != k

#     stat_jk_bottom_red[k, :], stat_jk_bottom_blue[k, :], \
#     stat_jk_top_red[k, :]   , stat_jk_top_blue[k, :]     = Ng8_percentile_env_LF(gal_pos[jk_mask], gal_Mr[jk_mask], \
#                                                                                  gal_gmr[jk_mask], gal_z[jk_mask], Mr_bins)
# # env_LF_top20_red   , env_LF_top20_blue     = Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, gal_fac, Mr_bins)

# output_dict['cov_top_red'] = np.cov(stat_jk_top_red.T, bias = 1) * (n_jk - 1)
# output_dict['cov_top_blue'] = np.cov(stat_jk_top_blue.T, bias = 1) * (n_jk - 1)
# output_dict['cov_bottom_red'] = np.cov(stat_jk_bottom_red.T, bias = 1) * (n_jk - 1)
# output_dict['cov_bottom_blue'] = np.cov(stat_jk_bottom_blue.T, bias = 1) * (n_jk - 1)
# output_dict_jk['jk_sample_top_red'] = stat_jk_top_red
# output_dict_jk['jk_sample_top_blue'] = stat_jk_top_blue
# output_dict_jk['jk_sample_bottom_red'] = stat_jk_bottom_red
# output_dict_jk['jk_sample_bottom_blue'] = stat_jk_bottom_blue

# np.savez('./SDSS_bright1_Mr_C_Ng8_env_LF.npz', **output_dict)
# np.savez('./SDSS_bright1_Mr_C_Ng8_env_LF_jk_samples.npz', **output_dict_jk)

In [ ]:
# data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data

# gal_pos = np.array([data['x']%1000,
#                     data['y']%1000,    
#                     data['z']%1000]).T
# gal_Mr  = np.array(data['Mr'])
# gal_gmr = np.array(data['gmr'])

# CCMD_box_env_LF_bottom20_red, CCMD_box_env_LF_bottom20_blue, \
# CCMD_box_env_LF_top20_red   , CCMD_box_env_LF_top20_blue     = Ng8_percentile_env_LF_theory(gal_pos, gal_Mr, gal_gmr, Mr_bins)

# np.save('./CCMD_box_env_LF_bottom20_red.npy' , CCMD_box_env_LF_bottom20_red )
# np.save('./CCMD_box_env_LF_bottom20_blue.npy', CCMD_box_env_LF_bottom20_blue)
# np.save('./CCMD_box_env_LF_top20_red.npy'    , CCMD_box_env_LF_top20_red    )
# np.save('./CCMD_box_env_LF_top20_blue.npy'   , CCMD_box_env_LF_top20_blue   )

In [ ]:
# data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data

# gal_pos_rsd =   np.array([  data['x']%1000,
#                             data['y']%1000,    
#                             (data['z'] + data['vz']/100)%1000]).T # vz/100 [Mpc/h], H0 = 100h [km/s/Mpc]

# gal_Mr  = np.array(data['Mr'])
# gal_gmr = np.array(data['gmr'])

# CCMD_box_rsd_env_LF_bottom20_red, CCMD_box_rsd_env_LF_bottom20_blue, \
# CCMD_box_rsd_env_LF_top20_red   , CCMD_box_rsd_env_LF_top20_blue     = Ng8_percentile_env_LF_theory(gal_pos_rsd, gal_Mr, gal_gmr, Mr_bins)

# np.save('./CCMD_box_rsd_env_LF_bottom20_red.npy' , CCMD_box_rsd_env_LF_bottom20_red )
# np.save('./CCMD_box_rsd_env_LF_bottom20_blue.npy', CCMD_box_rsd_env_LF_bottom20_blue)
# np.save('./CCMD_box_rsd_env_LF_top20_red.npy'    , CCMD_box_rsd_env_LF_top20_red    )
# np.save('./CCMD_box_rsd_env_LF_top20_blue.npy'   , CCMD_box_rsd_env_LF_top20_blue   )


In [ ]:
def cov_to_corr(input_stat, cov):
    variances = np.diag(cov) 
    zero_element = variances == 0
    zero_elements_indices = np.where(variances == 0)[0]
    input_stat = input_stat[~zero_element]
    # bestfit = bestfit[~zero_element]
    ### Remove the ith row and column
    cov = np.delete(np.delete(cov, zero_elements_indices, axis=0), zero_elements_indices, axis=1)
    ### Normalize the covariance matrix
    stdv = np.sqrt(np.diag(cov))  # Square root of the diagonal elements
    outer_diag = np.outer(stdv, stdv)   # Outer product of the diagonal elements
    corr = cov / outer_diag
    return input_stat, corr, stdv

In [ ]:
### CCMD 32 Quads
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

bot_red_data  = []
bot_blue_data = []
top_red_data  = []
top_blue_data = []

for var in vars:
    for quad in quads:
        bottom_red_temp  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_bottom20_red_env_LF.npy')
        bot_red_data.append(bottom_red_temp)
        bottom_blue_temp = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_bottom20_blue_env_LF.npy')
        bot_blue_data.append(bottom_blue_temp)  

        top_red_temp     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_top20_red_env_LF.npy')
        top_red_data.append(top_red_temp)
        top_blue_temp    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_top20_blue_env_LF.npy')
        top_blue_data.append(top_blue_temp)

bot_red_data  = np.array(bot_red_data)
bot_blue_data = np.array(bot_blue_data)
top_red_data  = np.array(top_red_data)
top_blue_data = np.array(top_blue_data)

original_statistics_bot_red  = np.mean(bot_red_data , axis=0)
original_statistics_bot_blue = np.mean(bot_blue_data, axis=0)
original_statistics_top_red  = np.mean(top_red_data , axis=0)
original_statistics_top_blue = np.mean(top_blue_data, axis=0)
# bottom_cov_matrix          = np.cov(bottom_data, rowvar=False)  # rowvar=False, column for variants
# top_cov_matrix             = np.cov(top_data, rowvar=False)  # rowvar=False, column for variants

bot_red_std           = np.std(bot_red_data , axis=0)  # rowvar=False, column for variants
bot_blue_std          = np.std(bot_blue_data, axis=0)  # rowvar=False, column for variants
top_red_std           = np.std(top_red_data , axis=0)  # rowvar=False, column for variants
top_blue_std          = np.std(top_blue_data, axis=0)  # rowvar=False, column for variants

np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' , original_statistics_bot_red )
np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy', original_statistics_bot_blue)
np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_mean.npy'    , original_statistics_top_red )
np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   , original_statistics_top_blue)

np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_std.npy' , bot_red_std )
np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_std.npy', bot_blue_std)
np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_std.npy'    , top_red_std )
np.save('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_std.npy'   , top_blue_std)

### Plot

In [ ]:
SDSS_env_LF_bot_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/SDSS_bright1_Mr_C_Ng8_bottom20_red_env_LF.npy' )
SDSS_env_LF_bot_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/SDSS_bright1_Mr_C_Ng8_bottom20_blue_env_LF.npy')
SDSS_env_LF_top_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/SDSS_bright1_Mr_C_Ng8_top20_red_env_LF.npy'    )
SDSS_env_LF_top_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/SDSS_bright1_Mr_C_Ng8_top20_blue_env_LF.npy'   )

# SDSS_env_LF_cov      = np.load('./SDSS_bright1_Mr_C_Ng8_env_LF.npz')
# SDSS_env_LF_jk_sample= np.load('./SDSS_bright1_Mr_C_Ng8_env_LF_jk_samples.npz')


CCMD_env_LF_mean_bot_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' )
CCMD_env_LF_mean_bot_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy')
CCMD_env_LF_mean_top_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_mean.npy'    )
CCMD_env_LF_mean_top_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   )

CCMD_env_LF_std_bot_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_std.npy' )
CCMD_env_LF_std_bot_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_std.npy')
CCMD_env_LF_std_top_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_std.npy'    )
CCMD_env_LF_std_top_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_area_limited/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_std.npy'   )

In [ ]:
# ### SDSS
# output_dict_bottom_err16 = collections.defaultdict(list)
# output_dict_bottom_err84 = collections.defaultdict(list)
# output_dict_top_err16    = collections.defaultdict(list)
# output_dict_top_err84    = collections.defaultdict(list)

# for j in range(4): # Mr, z
#     for i in range(4): # Color

#         bottom_percentile_16 = np.percentile(SDSS_cov_Mr_C_bottom20_jk, 16, axis=0)
#         bottom_percentile_84 = np.percentile(SDSS_cov_Mr_C_bottom20_jk, 84, axis=0)
#         output_dict_bottom_err16['err16_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = bottom_percentile_16
#         output_dict_bottom_err84['err84_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = bottom_percentile_84
        
#         top_percentile_16 = np.percentile(SDSS_cov_Mr_C_top20_jk, 16, axis=0)
#         top_percentile_84 = np.percentile(SDSS_cov_Mr_C_top20_jk, 84, axis=0)
#         output_dict_top_err16['err16_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = top_percentile_16
#         output_dict_top_err84['err84_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = top_percentile_84

#     np.savez('./SDSS_bright1_Mr_C_Ng8_bottom20_pimax_60_60_60_40_err16.npz', **output_dict_bottom_err16)
#     np.savez('./SDSS_bright1_Mr_C_Ng8_bottom20_pimax_60_60_60_40_err84.npz', **output_dict_bottom_err84)

#     np.savez('./SDSS_bright1_Mr_C_Ng8_top20_pimax_60_60_60_40_err16.npz', **output_dict_top_err16)
#     np.savez('./SDSS_bright1_Mr_C_Ng8_top20_pimax_60_60_60_40_err84.npz', **output_dict_top_err84)

In [ ]:
# 16-84% error

# CCMD_env_LF_bot_red_percentile_16  = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_err16.npy'  )
# CCMD_env_LF_bot_red_percentile_84  = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_err84.npy'  )
# CCMD_env_LF_bot_blue_percentile_16 = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_err16.npy' )
# CCMD_env_LF_bot_blue_percentile_84 = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_err84.npy' )

# CCMD_env_LF_top_red_percentile_16  = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_err16.npy' )
# CCMD_env_LF_top_red_percentile_84  = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_err84.npy' )
# CCMD_env_LF_top_blue_percentile_16 = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_err16.npy')
# CCMD_env_LF_top_blue_percentile_84 = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_err84.npy')


# SDSS_bright1_bottom20_err16 = np.load('./SDSS_bright1_Mr_C_Ng8_bottom20_pimax_60_60_60_40_err16.npz')
# SDSS_bright1_bottom20_err84 = np.load('./SDSS_bright1_Mr_C_Ng8_bottom20_pimax_60_60_60_40_err84.npz')

# SDSS_bright1_top20_err16    = np.load('./SDSS_bright1_Mr_C_Ng8_top20_pimax_60_60_60_40_err16.npz')
# SDSS_bright1_top20_err84    = np.load('./SDSS_bright1_Mr_C_Ng8_top20_pimax_60_60_60_40_err84.npz')

In [ ]:
CCMD_box_env_LF_bottom20_red  = np.load('./CCMD_box_env_LF_bottom20_red.npy' )
CCMD_box_env_LF_bottom20_blue = np.load('./CCMD_box_env_LF_bottom20_blue.npy')
CCMD_box_env_LF_top20_red     = np.load('./CCMD_box_env_LF_top20_red.npy'    )
CCMD_box_env_LF_top20_blue    = np.load('./CCMD_box_env_LF_top20_blue.npy'   )

In [ ]:
CCMD_box_rsd_env_LF_bottom20_red  = np.load('./CCMD_box_rsd_env_LF_bottom20_red.npy' )
CCMD_box_rsd_env_LF_bottom20_blue = np.load('./CCMD_box_rsd_env_LF_bottom20_blue.npy')
CCMD_box_rsd_env_LF_top20_red     = np.load('./CCMD_box_rsd_env_LF_top20_red.npy'    )
CCMD_box_rsd_env_LF_top20_blue    = np.load('./CCMD_box_rsd_env_LF_top20_blue.npy'   )

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (10, 6), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

### red ##################################################################################################################################
# stat = CCMD_32Quads_bottom20['CLF_bot20_red']
# bottom_err16 = CCMD_32Quads_bottom20_err16['bot20_red_err16']
# bottom_err84 = CCMD_32Quads_bottom20_err84['bot20_red_err84']
# yerr = np.sqrt(CCMD_32Quads_bottom20_cov['wprp_bottom20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())
# positive value

### label ###########################################################################################################################################
ax[0].plot(Mr_bins_x,  CCMD_env_LF_mean_bot_red - 1, color = 'black', alpha = 0.9, lw = 3, ls = '--', label = 'CCMD')
stat = CCMD_env_LF_mean_bot_red
yerr = CCMD_env_LF_std_bot_red
ax[0].errorbar( Mr_bins_x, stat-1, yerr = yerr, color = 'black', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'black', mew = 2.5, mfc = 'white', label = 'SDSS')

ax[1].plot(Mr_bins_x,  CCMD_env_LF_mean_bot_red - 1, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--' , label = 'Ng8 bot20%')
ax[1].plot(Mr_bins_x,  CCMD_env_LF_mean_top_red - 1, color = 'orange', alpha = 0.9, lw = 3, ls = '--' , label = 'Ng8 top20%')

### bottom 20% red ##################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_red
yerr = CCMD_env_LF_std_bot_red
ax[1].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# SDSS
stat = SDSS_env_LF_bot_red
# yerr = np.sqrt(SDSS_env_LF_cov['cov_bottom_red'].diagonal())
# ax[1].plot(Mr_bins_x[:], stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[1].scatter(Mr_bins_x[:], stat[:], facecolor = 'white', edgecolor = 'forestgreen', alpha = 0.9, marker = 'o')
# ax[1].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'forestgreen', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
#                 marker = 'o', ms = 6, mec = 'forestgreen', mew = 2.5, mfc = 'white')
# CCMD box
# ax[1].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '--')
# ax[1].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '--')

### top 20% red ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_red
yerr = CCMD_env_LF_std_top_red
ax[1].plot(Mr_bins_x[:], stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
# SDSS
stat = SDSS_env_LF_top_red
# yerr = np.sqrt(SDSS_env_LF_cov['cov_top_red'].diagonal())
# ax[1].plot(Mr_bins_x[:],  stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '-' , label = 'SDSS')
ax[1].scatter(Mr_bins_x[:],  stat[:], facecolor = 'white', edgecolor = 'orange', alpha = 0.9, marker = 'o' , label = 'SDSS')
# ax[1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
# ax[1].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
#                 marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')
# CCMD box
# ax[1].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box ')
# ax[1].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box rsd')

### bottom 20% blue ###############################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_blue
yerr = CCMD_env_LF_std_bot_blue
ax[0].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# SDSS
stat = SDSS_env_LF_bot_blue
# yerr = np.sqrt(SDSS_env_LF_cov['cov_bottom_blue'].diagonal())
# ax[0].plot(Mr_bins_x[:], stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[0].scatter(Mr_bins_x[:], stat[:], facecolor = 'white', edgecolor = 'forestgreen', alpha = 0.9, marker = 'o',)
# ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
# ax[0].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'forestgreen', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
#                 marker = 'o', ms = 6, mec = 'forestgreen', mew = 2.5, mfc = 'white')
# CCMD box
# ax[0].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '--')
# ax[0].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '--')

### top 20% blue ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_blue
yerr = CCMD_env_LF_std_top_blue
ax[0].plot(Mr_bins_x[:],  stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
# SDSS
stat = SDSS_env_LF_top_blue
# yerr = np.sqrt(SDSS_env_LF_cov['cov_top_blue'].diagonal())
# ax[0].plot(Mr_bins_x[:],  stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '-')
ax[0].scatter(Mr_bins_x[:],  stat[:], facecolor = 'white', edgecolor = 'orange', alpha = 0.9, marker = 'o')
# ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
# ax[0].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
#                 marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')
# CCMD box
# ax[0].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '-')
# ax[0].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '-')

####################################################################################################################################################   
# ax[0].text(3.7+x_add[i],1.3e3, Co_thr_text[j,i], fontsize = 16)
ax[0].text(-20.1, 0.005, 'blue galaxies', color = 'k', fontsize = 16)
ax[1].text(-20.2, 0.005, 'red galaxies' , color = 'k', fontsize = 16)
# ax[0].text(-19.8, 0.0025, r'DDP: $M_r < -19$', color = 'k', fontsize = 16)
# ax[1].text(-19.8, 0.0025, r'DDP: $M_r < -19$', color = 'k', fontsize = 16)

ax[0].set_ylabel(r'$\log(\Phi_{N_g^8}/h^3 \rm Mpc^{-3}mag^{-1})$',fontsize = 24,)

for i in range(2):
    # ax[i].set_xlabel(r'$M_r$',fontsize = 20,)
    ax[i].set_xlim(-22.5, -17.5)
    ax[i].set_ylim(10**-6.1, 10**-1.9)
    ax[i].set_yscale('log')
    ax[i].tick_params(axis = 'x', direction = 'in', which = u'both')
    ax[i].tick_params(axis = 'y', direction = 'in', which = u'both')

legend = ax[1].legend(fontsize = 16, loc = 'lower left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('orange')
legend = ax[0].legend(fontsize = 16, loc = 'lower left',frameon = False) 
# ax[0].text(-18, 0.003, '100 < RA < 300')#1/fac, 
# ax[1].text(-18, 0.003, '100 < RA < 300')#1/fac, 

fig.supxlabel(r'$M_r$', fontsize=24, fontweight='bold', y = -0.02)

plt.gca().invert_xaxis()

# plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize = (10, 6), sharex = True, sharey = True)
# plt.subplots_adjust(hspace = 0, wspace = 0)

# pimax_texts    = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']
# x_add = [1.5, -0.65, -0.65, 0.1]


# ### red ##################################################################################################################################
# # stat = CCMD_32Quads_bottom20['CLF_bot20_red']
# # bottom_err16 = CCMD_32Quads_bottom20_err16['bot20_red_err16']
# # bottom_err84 = CCMD_32Quads_bottom20_err84['bot20_red_err84']
# # yerr = np.sqrt(CCMD_32Quads_bottom20_cov['wprp_bottom20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())
# # positive value

# ### label ###########################################################################################################################################
# ax[1].plot(Mr_bins_x,  CCMD_env_LF_mean_bot_red - 1, color = 'black', alpha = 0.9, lw = 3, ls = '--', label = 'Ng8 bot20%')
# ax[1].plot(Mr_bins_x,  CCMD_env_LF_mean_top_red - 1, color = 'black', alpha = 0.9, lw = 3, ls = '-' , label = 'Ng8 top20%')

# n = 5
# ### bottom 20% red ##################################################################################################################################
# stat = CCMD_env_LF_mean_bot_red
# yerr = CCMD_env_LF_std_bot_red
# # ax[0].plot(Mr_bins_x[:-n], stat[:-n], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')#, label = 'CCMD Ng8 bot20%')
# # ax[0].fill_between(Mr_bins_x[:-n], stat[:-n] - yerr[:-n], stat[:-n] + yerr[:-n], color = 'forestgreen', alpha = 0.3)
# # # ax[0].fill_between(Mr_bins_x,  CCMD_env_LF_bot_red_percentile_16, CCMD_env_LF_bot_red_percentile_84, color = 'forestgreen', alpha = 0.3)
# # ax[0].plot(Mr_bins_x[:-n],  SDSS_env_LF_bot_red[:-n], color = 'orange', alpha = 0.9, lw = 3, ls = '--')#, label = 'SDSS Ng8 bot20%')
# ax[0].plot(Mr_bins_x[:-n],  CCMD_box_env_LF_bottom20_red[:-n], color = 'black' , alpha = 0.9, lw = 3, ls = '--')#, label = 'SDSS Ng8 bot20%')
# ax[0].plot(Mr_bins_x[:-n],  CCMD_box_rsd_env_LF_bottom20_red[:-n], color = 'red' , alpha = 0.9, lw = 3, ls = '--')#, label = 'SDSS Ng8 bot20%')

# ### top 20% red #####################################################################################################################################
# stat = CCMD_env_LF_mean_top_red
# yerr = CCMD_env_LF_std_top_red
# # ax[0].plot(Mr_bins_x[:-n], stat[:-n], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
# # ax[0].fill_between(Mr_bins_x[:-n], stat[:-n] - yerr[:-n], stat[:-n] + yerr[:-n], color = 'forestgreen', alpha = 0.3)
# # # ax[0].fill_between(Mr_bins_x,  CCMD_env_LF_top_red_percentile_16, CCMD_env_LF_top_red_percentile_84, color = 'forestgreen', alpha = 0.3)
# # ax[0].plot(Mr_bins_x[:-n],  SDSS_env_LF_top_red[:-n], color = 'orange', alpha = 0.9, lw = 3, ls = '-' , label = 'SDSS')
# ax[0].plot(Mr_bins_x[:-n],  CCMD_box_env_LF_top20_red[:-n], color = 'black' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box')
# ax[0].plot(Mr_bins_x[:-n],  CCMD_box_rsd_env_LF_top20_red[:-n], color = 'red' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box rsd')

# ### bottom 20% blue ###################################################################################################################################
# stat = CCMD_env_LF_mean_bot_blue
# yerr = CCMD_env_LF_std_bot_blue
# # ax[1].plot(Mr_bins_x[:-n],  stat[:-n], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')#, label = 'CCMD Ng8 bot20%')
# # ax[1].fill_between(Mr_bins_x[:-n], stat[:-n] - yerr[:-n], stat[:-n] + yerr[:-n], color = 'forestgreen', alpha = 0.3)
# # # ax[1].fill_between(Mr_bins_x,  CCMD_env_LF_bot_blue_percentile_16, CCMD_env_LF_bot_blue_percentile_84, color = 'forestgreen', alpha = 0.3)
# # ax[1].plot(Mr_bins_x[:-n],  SDSS_env_LF_bot_blue[:-n], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
# ax[1].plot(Mr_bins_x[:-n],  CCMD_box_env_LF_bottom20_blue[:-n], color = 'black', alpha = 0.9, lw = 3, ls = '--')
# ax[1].plot(Mr_bins_x[:-n],  CCMD_box_rsd_env_LF_bottom20_blue[:-n], color = 'red', alpha = 0.9, lw = 3, ls = '--')

# ### top 20% blue ###################################################################################################################################
# stat = CCMD_env_LF_mean_top_blue
# yerr = CCMD_env_LF_std_top_blue
# # ax[1].plot(Mr_bins_x[:-n],  stat[:-n], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')#, label = 'CCMD Ng8 top20%')
# # ax[1].fill_between(Mr_bins_x[:-n], stat[:-n] - yerr[:-n], stat[:-n] + yerr[:-n], color = 'forestgreen', alpha = 0.3)
# # # ax[1].fill_between(Mr_bins_x,  CCMD_env_LF_top_blue_percentile_16, CCMD_env_LF_top_blue_percentile_84, color = 'forestgreen', alpha = 0.3)
# # ax[1].plot(Mr_bins_x[:-n],  SDSS_env_LF_top_blue[:-n], color = 'orange', alpha = 0.9, lw = 3, ls = '-' )
# ax[1].plot(Mr_bins_x[:-n],  CCMD_box_env_LF_top20_blue[:-n], color = 'black', alpha = 0.9, lw = 3, ls = '-')
# ax[1].plot(Mr_bins_x[:-n],  CCMD_box_rsd_env_LF_top20_blue[:-n], color = 'red', alpha = 0.9, lw = 3, ls = '-')

# ############################################################################################################################################   
# # ax[0].text(3.7+x_add[i],1.3e3, Co_thr_text[j,i], fontsize = 16)
# ax[0].text(-21.5, 10**-2.4, 'red' , color = 'red' , fontsize = 18)
# ax[1].text(-21.5, 10**-2.4, 'blue', color = 'blue', fontsize = 18)

# ax[0].set_ylabel(r'$\log(\Phi_{N_g^8}/h^3 \rm Mpc^{-3}mag^{-1})$',fontsize = 18,)

# for i in range(2):
#     ax[i].set_xlabel(r'$M_r$',fontsize = 18,)
#     ax[i].set_xlim(-22.5, -17.5)
#     ax[i].set_ylim(10**-6.1, 10**-1.9)
#     ax[i].set_yscale('log')
#     ax[i].tick_params(axis = 'x', direction = 'in', which = u'both')
#     ax[i].tick_params(axis = 'y', direction = 'in', which = u'both')

# legend = ax[0].legend(fontsize = 16, loc = 'lower left',frameon = False) 
# legend.get_texts()[0].set_color('forestgreen')
# legend.get_texts()[1].set_color('orange')
# legend = ax[1].legend(fontsize = 16, loc = 'lower left',frameon = False) 

# plt.gca().invert_xaxis()

# plt.suptitle(r'$z_{cut}: Mr = -19$', y =0.95)
# ax[0].text(-18, 0.006, r'$z_{cut}: Mr = -19$')
# ax[1].text(-18, 0.006, r'$z_{cut}: Mr = -19$')
# # plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (10, 6), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

ax[1].plot(Mr_bins_x[:], CCMD_env_LF_mean_bot_red[:] - 1, color = 'k', alpha = 0.9, lw = 3, ls = '--', label = 'Ng8 bot20%')
ax[1].plot(Mr_bins_x[:], CCMD_env_LF_mean_top_red[:] - 1, color = 'k', alpha = 0.9, lw = 3, ls = '-' , label = 'Ng8 top20%')

### bottom 20% red ##################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_red
yerr = CCMD_env_LF_std_bot_red
ax[1].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# CCMD box
ax[1].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '--')
ax[1].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '--')

### top 20% red ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_red
yerr = CCMD_env_LF_std_top_red
ax[1].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')
ax[1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)

# CCMD box
ax[1].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '-')
ax[1].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '-')

### bottom 20% blue ###############################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_blue
yerr = CCMD_env_LF_std_bot_blue
ax[0].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)

# CCMD box
ax[0].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '--')
ax[0].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '--')

### top 20% blue ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_blue
yerr = CCMD_env_LF_std_top_blue
ax[0].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)

# CCMD box
ax[0].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/o rsd')
ax[0].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/ rsd')

####################################################################################################################################################   
# ax[0].text(3.7+x_add[i],1.3e3, Co_thr_text[j,i], fontsize = 16)
ax[0].text(-20.1, 0.005, 'blue galaxies', color = 'k', fontsize = 16)
ax[1].text(-20.2, 0.005, 'red galaxies' , color = 'k', fontsize = 16)
ax[0].text(-19.8, 0.0025, r'DDP: $M_r < -19$', color = 'k', fontsize = 16)
ax[1].text(-19.8, 0.0025, r'DDP: $M_r < -19$', color = 'k', fontsize = 16)

ax[0].set_ylabel(r'$\log(\Phi_{N_g^8}/h^3 \rm Mpc^{-3}mag^{-1})$',fontsize = 24,)

for i in range(2):
    # ax[i].set_xlabel(r'$M_r$',fontsize = 20,)
    ax[i].set_xlim(-22.5, -17.5)
    ax[i].set_ylim(10**-6.1, 10**-1.9)
    ax[i].set_yscale('log')
    ax[i].tick_params(axis = 'x', direction = 'in', which = u'both')
    ax[i].tick_params(axis = 'y', direction = 'in', which = u'both')

legend = ax[0].legend(fontsize = 16, loc = 'lower left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[2].set_color('red')
legend = ax[1].legend(fontsize = 16, loc = 'lower left',frameon = False) 
ax[0].text(-18, 1e-4, 'Boundary limited', fontsize = 16)#1/fac, '100 < RA < 300'
ax[1].text(-18, 1e-4, 'Boundary limited', fontsize = 16)#1/fac, '100 < RA < 300'

fig.supxlabel(r'$M_r$', fontsize=24, fontweight='bold', y = -0.02)

plt.gca().invert_xaxis()

# plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')